# Profils test HEC (étudiant + ambassadeur)

Dans les CSV mock, **HEC** est la seule école commune entre `ecole_actuelle` (étudiants) et `nom_etablissement` (ambassadeurs) pour alimenter le graphe Linkage (`School` + `STUDIES_AT` / `STUDIED_AT`).

Ce notebook fixe **deux lignes réelles** du CSV pour rejouer les requêtes Neo4j / API sans chercher au hasard.

In [ ]:
from pathlib import Path
import pandas as pd

# Résolution du dossier Résultats (notebook dans moc_data/scrypte/)
_here = Path.cwd()
for DATA in (_here.parent / "Résultats", _here / "Résultats", Path("Site/moc_data/Résultats")):
    if (DATA / "database_etudiants_300k.csv").exists():
        break
else:
    raise FileNotFoundError("Place le kernel dans scrypte ou à la racine du repo pour trouver les CSV.")

stu = pd.read_csv(
    DATA / "database_etudiants_300k.csv",
    usecols=["id", "prenom", "nom", "email", "ville", "niveau_actuel", "ecole_actuelle", "source_lead"],
)
amb = pd.read_csv(
    DATA / "ambassadeurs_data_heavy.csv",
    usecols=[
        "id", "prenom", "nom", "email", "ville", "niveau_actuel",
        "nom_etablissement", "type_etablissement", "score_attractivite",
    ],
)

# --- Profil étudiant "sympa" (gars, HEC, niveau crédible) ---
STUDENT_ID = 114
s = stu.loc[stu["id"] == STUDENT_ID].iloc[0]

# --- Ambassadeur "cool" (HEC, meilleur score_attractivite du fichier) ---
AMB_ID = 388
a = amb.loc[amb["id"] == AMB_ID].iloc[0]

display(pd.DataFrame([s]).T.rename(columns={0: "valeur"}))
display(pd.DataFrame([a]).T.rename(columns={0: "valeur"}))

print("--- Résumé pour tests Linkage / Neo4j ---")
print(f"Étudiant (center): email={s['email']!r}  |  School STUDIES_AT: {s['ecole_actuelle']!r}")
print(f"Ambassadeur:      email={a['email']!r}  |  School STUDIED_AT: {a['nom_etablissement']!r}")
print("Même nom d'école 'HEC' → même nœud School si l'import normalise bien les chaînes.")

**Profil étudiant** — **Léo Martin** (`id=114`) : Bac+3, **HEC**, Toulouse, lead Instagram. Email : `léo.martin114@mail.fr`.

**Profil ambassadeur** — **Hugo Garcia** (`id=388`) : **HEC**, Commerce, Licence 2, Bordeaux, **score_attractivite 96** (un des profils les plus « visibles » du fichier). Email : `hugo.garcia388@etudiant.fr`.

> Le générateur étudiant ne distingue pas « veut aller à HEC » et « est à HEC » : ici on utilise **`ecole_actuelle == "HEC"`** pour que le même libellé matche **`nom_etablissement`** côté ambassadeurs après import.